---
title: "Data Structures and Algorithms: Network Flow and Circulations"
lang: en
format:
  html:
    toc: true
    toc-depth: 5
    theme: cosmo
    # code-fold: true
jupyter: python
---


[Back to Data Structures and Algorithms guideline](Data-Structure&Algorithm.html)


## **Network Flow and Circulations** {#network-flow-and-circulations}

**Network flow** models a divisible resource moving through a directed network with limited carrying capacity. The resource may be water through pipes, packets through links, vehicles through roads, jobs assigned to workers, or abstract units representing compatible choices. Unlike a shortest-path problem, which asks for one route, a flow problem asks how many units can travel simultaneously across many interacting routes.

The central idea is surprisingly compact: local capacity and conservation rules define which assignments are legal, while a global objective measures how much reaches the destination. A residual graph then records both unused capacity and the ability to revise earlier routing decisions. That reversible representation is what turns a greedy-looking sequence of path choices into a correct optimization method.

This chapter uses one running network with source $s$, sink $t$, and intermediate vertices $a$ and $b$:

- $s\rightarrow a$ has capacity $4$ and $s\rightarrow b$ has capacity $2$;
- $a\rightarrow b$ has capacity $1$;
- $a\rightarrow t$ has capacity $2$ and $b\rightarrow t$ has capacity $3$.

The network can carry five units from $s$ to $t$. The same example will be viewed as a feasible flow, a residual graph, a sequence of augmentations, and a minimum cut so that the concepts remain connected.

::: {.callout-important}
Flow is not the same as reachability. Reachability asks whether at least one route exists. Maximum flow asks how much can be routed while all routes compete for shared capacities.
:::


### **Flow Networks and Feasible Flows** {#flow-networks-and-feasible-flows}

A **flow network** is a directed graph $G=(V,E)$ with a non-negative capacity $c(u,v)$ on every directed edge $(u,v)$. It has a distinguished **source** $s$, where flow originates, and **sink** $t$, where flow is collected. A flow assignment $f(u,v)$ tells us how much of the resource currently uses each edge.

Think of each edge as a pipe. Capacity describes the pipe's physical limit; flow describes the amount currently passing through it. Merely keeping each pipe below its limit is not enough. An ordinary intermediate junction cannot create or destroy water, so everything entering it must also leave it.

A flow is **feasible** when it satisfies two local constraints. First, every edge obeys the capacity constraint

$$
0 \le f(u,v) \le c(u,v).
$$

Here $(u,v)$ is a directed edge, $c(u,v)$ is its capacity, and $f(u,v)$ is its assigned flow. The lower bound prevents flow from moving backward on that edge; backward correction will instead be represented explicitly in the residual graph.

Second, every vertex other than $s$ and $t$ obeys flow conservation:

$$
\sum_{(u,v)\in E} f(u,v)
=
\sum_{(v,w)\in E} f(v,w),
\qquad v\in V\setminus\{s,t\}.
$$

The left sum is all flow entering $v$ and the right sum is all flow leaving $v$. The condition excludes $s$ and $t$ because the source is allowed to produce net outflow and the sink is allowed to absorb it.

The **value of the flow** is the net amount leaving the source, equivalently the net amount entering the sink:

$$
|f|
=
\sum_{(s,v)\in E}f(s,v)-\sum_{(v,s)\in E}f(v,s)
=
\sum_{(v,t)\in E}f(v,t)-\sum_{(t,v)\in E}f(t,v).
$$

The symbol $|f|$ is an objective value, not an absolute value applied edge by edge. Conservation makes the two expressions equal: flow cannot disappear between $s$ and $t$.

~~~text
CHECK-FEASIBLE-FLOW(capacity, flow, source, sink)
    for each directed edge (u,v)
        if flow(u,v) < 0 or flow(u,v) > capacity(u,v)
            return infeasible

    compute net inflow minus net outflow at every vertex
    if an intermediate vertex has nonzero balance
        return infeasible

    if source outflow equals sink inflow
        return feasible and that common flow value
    return infeasible
~~~

![A feasible flow on the running network, with edge labels shown as flow over capacity and the three required checks.](assets/flow-network-feasible-flow.svg){fig-align="center" width="100%"}

<details>
<summary>Python implementation: validate a flow assignment</summary>

~~~python
from collections import defaultdict


def check_feasible_flow(
    capacity: dict[tuple[str, str], int],
    flow: dict[tuple[str, str], int],
    source: str,
    sink: str,
) -> tuple[bool, int]:
    """Return (is_feasible, flow_value) for a directed flow assignment."""
    vertices = {source, sink}
    balance: dict[str, int] = defaultdict(int)  # inflow minus outflow

    # A nonzero flow on a missing edge is not part of this network.
    if any(edge not in capacity and amount != 0 for edge, amount in flow.items()):
        return False, 0

    for (u, v), edge_capacity in capacity.items():
        vertices.update((u, v))
        amount = flow.get((u, v), 0)

        # Check the edge-local capacity contract.
        if edge_capacity < 0 or amount < 0 or amount > edge_capacity:
            return False, 0

        balance[u] -= amount
        balance[v] += amount

    # Every internal vertex must have equal inflow and outflow.
    for vertex in vertices - {source, sink}:
        if balance[vertex] != 0:
            return False, 0

    value = -balance[source]
    is_feasible = value >= 0 and balance[sink] == value
    return is_feasible, value if is_feasible else 0


capacity = {
    ("s", "a"): 4,
    ("s", "b"): 2,
    ("a", "b"): 1,
    ("a", "t"): 2,
    ("b", "t"): 3,
}
flow = {
    ("s", "a"): 3,
    ("s", "b"): 2,
    ("a", "b"): 1,
    ("a", "t"): 2,
    ("b", "t"): 3,
}

assert check_feasible_flow(capacity, flow, "s", "t") == (True, 5)
print(check_feasible_flow(capacity, flow, "s", "t"))
~~~
</details>

The validator scans every edge and vertex once, so it takes $O(V+E)$ time and $O(V)$ auxiliary space for balances. The cost follows directly from the representation: every edge contributes once to two endpoint balances.

**Practice:** [LeetCode 1601 - Maximum Number of Achievable Transfer Requests](https://leetcode.com/problems/maximum-number-of-achievable-transfer-requests/) is not a maximum-flow implementation problem, but its "every building has zero net change" condition directly exercises flow conservation.


### **Residual Graphs and Augmenting Paths** {#residual-graphs-and-augmenting-paths}

A feasible flow records what has already been committed. To improve it, an algorithm needs a second representation that answers two questions: how much more can be sent forward, and how much earlier flow can be cancelled? The **residual graph** answers both.

For an original edge $(u,v)$, its forward residual capacity is

$$
c_f(u,v)=c(u,v)-f(u,v).
$$

The subscript $f$ indicates that the residual capacity depends on the current flow. If an edge of capacity $4$ carries $1$, then three additional units may still move forward.

The same edge also creates reverse residual capacity:

$$
c_f(v,u) \mathrel{+}= f(u,v).
$$

The reverse edge is not necessarily a physical edge in the original network. It represents permission to subtract previously routed flow from $(u,v)$. This is crucial: a path chosen early may later block a better combination, and the reverse edge lets a later augmenting path repair that choice without restarting.

An **augmenting path** is any $s$-to-$t$ path whose residual edges all have positive capacity. The most that can be added is its bottleneck

$$
\Delta=\min_{(u,v)\in P}c_f(u,v),
$$

where $P$ is the residual path and $\Delta$ is the augmentation amount. Every forward original edge on $P$ gains $\Delta$ flow; every reverse residual edge cancels $\Delta$ from its corresponding original edge.

~~~text
AUGMENT(capacity, flow, path P)
    build forward and reverse residual capacities
    delta <- minimum residual capacity on P

    for each residual edge (u,v) on P
        if (u,v) is an original forward edge
            flow(u,v) <- flow(u,v) + delta
        else
            flow(v,u) <- flow(v,u) - delta

    return delta and the updated flow
~~~

![One unit sent along a path consumes forward residual capacity and creates a reverse edge that can undo the decision.](assets/residual-augmentation.svg){fig-align="center" width="100%"}

<details>
<summary>Python implementation: build a residual graph and augment one path</summary>

~~~python
from collections import defaultdict


def residual_capacities(
    capacity: dict[tuple[str, str], int],
    flow: dict[tuple[str, str], int],
) -> dict[tuple[str, str], int]:
    residual: dict[tuple[str, str], int] = defaultdict(int)

    for (u, v), edge_capacity in capacity.items():
        amount = flow.get((u, v), 0)
        residual[(u, v)] += edge_capacity - amount  # unused forward capacity
        residual[(v, u)] += amount                  # cancellable flow

    return dict(residual)


def augment_path(
    capacity: dict[tuple[str, str], int],
    flow: dict[tuple[str, str], int],
    path: list[str],
) -> int:
    """Augment one residual path; assumes no antiparallel original edges."""
    residual = residual_capacities(capacity, flow)
    arcs = list(zip(path, path[1:]))

    if not arcs or any(residual.get(arc, 0) <= 0 for arc in arcs):
        raise ValueError("path is not an augmenting path")

    delta = min(residual[arc] for arc in arcs)
    for u, v in arcs:
        if (u, v) in capacity:
            flow[(u, v)] = flow.get((u, v), 0) + delta
        else:
            # Traversing a reverse residual edge cancels original flow.
            flow[(v, u)] -= delta

    return delta


capacity = {
    ("s", "a"): 4,
    ("s", "b"): 2,
    ("a", "b"): 1,
    ("a", "t"): 2,
    ("b", "t"): 3,
}
flow = {edge: 0 for edge in capacity}

assert augment_path(capacity, flow, ["s", "a", "b", "t"]) == 1
residual = residual_capacities(capacity, flow)
assert flow[("a", "b")] == 1
assert residual[("b", "a")] == 1  # The first choice can now be cancelled.
print(flow)
~~~
</details>

Building the residual graph takes $O(E)$ time and space. Updating a known path of length $k$ takes $O(k)$ time. Production implementations normally store paired forward and reverse edge objects, which also handle parallel and antiparallel original edges without ambiguity.

**Practice:** [LeetCode 2123 - Minimum Operations to Remove Adjacent Ones in Matrix](https://leetcode.com/problems/minimum-operations-to-remove-adjacent-ones-in-matrix/) is a premium problem whose standard bipartite-matching formulation relies on residual augmenting paths.


### **Maximum Flow and Minimum Cut** {#maximum-flow-and-minimum-cut}

A **maximum flow** is a feasible $s$-$t$ flow with the largest possible value. To prove that a candidate is maximum, it is not enough to say that one routing attempt cannot continue. We need a global certificate that no routing can carry more.

An $s$-$t$ **cut** partitions the vertices into two sets $(S,T)$ with $s\in S$ and $t\in T$. Its capacity counts original capacities directed from the source side to the sink side:

$$
c(S,T)=\sum_{u\in S,\,v\in T}c(u,v).
$$

The cut is a bottleneck boundary. Every unit reaching $t$ must cross from $S$ to $T$ at least once in net terms, so any feasible flow satisfies

$$
|f|\le c(S,T).
$$

This is **weak duality**: every cut is an upper bound on every flow. The **max-flow min-cut theorem** states that the best lower-side object and the best upper-side certificate meet exactly:

$$
\max_f |f|=\min_{(S,T)}c(S,T).
$$

After an augmenting-path algorithm stops, let $S$ be all vertices still reachable from $s$ through positive residual edges, and let $T=V\setminus S$. The sink is not in $S$, otherwise another augmenting path would exist. Every original edge from $S$ to $T$ is saturated, while every original edge from $T$ to $S$ carries zero flow. Therefore the current flow value equals the cut capacity, proving both are optimal.

~~~text
MIN-CUT-AFTER-MAX-FLOW(original_capacity, residual_graph, source)
    S <- all vertices reachable from source using positive residual edges
    T <- V minus S
    cut_edges <- original edges (u,v) with u in S and v in T
    return S, T, cut_edges, sum of their capacities
~~~

![A maximum flow and a cut with the same value, which certifies optimality.](assets/max-flow-min-cut-example.svg){fig-align="center" width="70%"}

*Visual source: [Chin Ho Lee, Max-flow min-cut example](https://commons.wikimedia.org/wiki/File:Max-flow_min-cut_example.svg), public domain. The unchanged SVG is stored locally for reliable rendering.*

<details>
<summary>Python implementation: recover a minimum cut from a final residual graph</summary>

~~~python
from collections import defaultdict, deque


def build_residual(
    capacity: dict[tuple[str, str], int],
    flow: dict[tuple[str, str], int],
) -> dict[str, dict[str, int]]:
    residual: dict[str, dict[str, int]] = defaultdict(lambda: defaultdict(int))
    for (u, v), edge_capacity in capacity.items():
        amount = flow.get((u, v), 0)
        residual[u][v] += edge_capacity - amount
        residual[v][u] += amount
    return {u: dict(neighbors) for u, neighbors in residual.items()}


def minimum_cut_from_flow(
    capacity: dict[tuple[str, str], int],
    flow: dict[tuple[str, str], int],
    source: str,
) -> tuple[set[str], set[str], int]:
    residual = build_residual(capacity, flow)
    vertices = {vertex for edge in capacity for vertex in edge}

    source_side = {source}
    queue = deque([source])
    while queue:
        u = queue.popleft()
        for v, remaining in residual.get(u, {}).items():
            if remaining > 0 and v not in source_side:
                source_side.add(v)
                queue.append(v)

    sink_side = vertices - source_side
    cut_capacity = sum(
        edge_capacity
        for (u, v), edge_capacity in capacity.items()
        if u in source_side and v in sink_side
    )
    return source_side, sink_side, cut_capacity


capacity = {
    ("s", "a"): 4,
    ("s", "b"): 2,
    ("a", "b"): 1,
    ("a", "t"): 2,
    ("b", "t"): 3,
}
maximum_flow = {
    ("s", "a"): 3,
    ("s", "b"): 2,
    ("a", "b"): 1,
    ("a", "t"): 2,
    ("b", "t"): 3,
}

source_side, sink_side, capacity_of_cut = minimum_cut_from_flow(
    capacity, maximum_flow, "s"
)
assert source_side == {"s", "a"}
assert sink_side == {"b", "t"}
assert capacity_of_cut == 5
print(source_side, sink_side, capacity_of_cut)
~~~
</details>

Once a maximum flow is known, residual reachability and cut extraction take $O(V+E)$ time and $O(V)$ traversal space. The expensive part is obtaining the maximum flow; the cut certificate itself is cheap to verify.

**Practice:** [LeetCode 1568 - Minimum Number of Days to Disconnect Island](https://leetcode.com/problems/minimum-number-of-days-to-disconnect-island/) is not a capacity-flow problem, but it develops the related habit of identifying a small separating cut rather than searching only for paths.


### **Ford-Fulkerson Algorithm** {#ford-fulkerson-algorithm}

**Ford-Fulkerson** is the general augmenting-path method for maximum flow. It repeatedly finds any residual $s$-$t$ path, pushes the path bottleneck, and updates paired residual capacities. It is a method rather than one fully specified traversal algorithm: DFS, BFS, or another search rule may choose the path.

The method improves on sending flow greedily through original edges because it searches the residual graph. Reverse residual edges allow a later path to reroute earlier flow, so a locally inconvenient first path does not permanently destroy the optimum.

~~~text
FORD-FULKERSON(G, capacity, source, sink)
    flow <- zero on every original edge
    residual <- capacity plus zero-capacity reverse edges

    while residual contains an s-to-t path P
        delta <- minimum residual capacity on P
        for each residual edge (u,v) in P
            residual(u,v) <- residual(u,v) - delta
            residual(v,u) <- residual(v,u) + delta
        flow_value <- flow_value + delta

    return flow_value and the flow encoded by the residual graph
~~~

![An animation of Ford-Fulkerson repeatedly selecting and augmenting residual paths.](assets/ford-fulkerson-animation.gif){fig-align="center" width="58%"}

*Visual source: [Xav65, FordFulkerson animation](https://commons.wikimedia.org/wiki/File:FordFulkerson.gif), licensed under CC BY-SA 4.0. The unchanged 10-frame GIF is stored locally.*

<details>
<summary>Python implementation: Ford-Fulkerson with depth-first path search</summary>

~~~python
def ford_fulkerson_dfs(
    vertex_count: int,
    edges: list[tuple[int, int, int]],
    source: int,
    sink: int,
) -> int:
    residual = [dict() for _ in range(vertex_count)]
    for u, v, capacity in edges:
        if capacity < 0:
            raise ValueError("capacity must be non-negative")
        residual[u][v] = residual[u].get(v, 0) + capacity
        residual[v].setdefault(u, 0)

    def send_flow(u: int, available: int, visited: set[int]) -> int:
        if u == sink:
            return available
        visited.add(u)

        for v, remaining in list(residual[u].items()):
            if remaining <= 0 or v in visited:
                continue

            pushed = send_flow(v, min(available, remaining), visited)
            if pushed > 0:
                # Consume forward residual capacity and create reverse capacity.
                residual[u][v] -= pushed
                residual[v][u] = residual[v].get(u, 0) + pushed
                return pushed
        return 0

    total_flow = 0
    while True:
        pushed = send_flow(source, float("inf"), set())
        if pushed == 0:
            return total_flow
        total_flow += pushed


edges = [
    (0, 1, 4),  # s -> a
    (0, 2, 2),  # s -> b
    (1, 2, 1),  # a -> b
    (1, 3, 2),  # a -> t
    (2, 3, 3),  # b -> t
]
assert ford_fulkerson_dfs(4, edges, 0, 3) == 5
print(ford_fulkerson_dfs(4, edges, 0, 3))
~~~
</details>

With integer capacities, every augmentation increases total flow by at least one. If $F$ is the final maximum-flow value and path search costs $O(E)$, the DFS version has the pseudo-polynomial bound $O(EF)$. It is called pseudo-polynomial because $F$ depends on the numeric capacity values, not merely on how many bits encode them. With irrational capacities and arbitrary path choices, the method can even fail to terminate; this motivates a deterministic path rule.

**Practice:** [LeetCode 1820 - Maximum Number of Accepted Invitations](https://leetcode.com/problems/maximum-number-of-accepted-invitations/) is a premium bipartite-matching problem that can be solved by unit-capacity Ford-Fulkerson augmentations.


### **Edmonds-Karp Algorithm** {#edmonds-karp-algorithm}

**Edmonds-Karp** is Ford-Fulkerson with one decisive rule: use breadth-first search to choose an augmenting path with the fewest residual edges. It does not necessarily choose the path with the largest bottleneck. Its benefit is predictable progress in the residual graph.

After each augmentation, the BFS distance from $s$ to every reachable vertex never decreases. When an edge becomes the bottleneck on a shortest augmenting path, that edge cannot become critical again until its source is reached at a strictly greater BFS level. This monotonicity limits how often edges can cause augmentations.

~~~text
EDMONDS-KARP(G, capacity, source, sink)
    initialize paired forward and reverse residual capacities
    total <- 0

    repeat
        use BFS on positive residual edges to record each parent
        if sink was not reached: return total

        reconstruct the shortest-edge path from sink to source
        delta <- minimum residual capacity on that path
        update forward and reverse residual capacities by delta
        total <- total + delta
~~~

![Breadth-first search finds two two-edge augmenting paths and then one three-edge path on the running network.](assets/edmonds-karp-bfs.svg){fig-align="center" width="100%"}

<details>
<summary>Python implementation: Edmonds-Karp with path reconstruction</summary>

~~~python
from collections import deque


def edmonds_karp(
    vertex_count: int,
    edges: list[tuple[int, int, int]],
    source: int,
    sink: int,
) -> tuple[int, list[tuple[list[int], int]]]:
    residual = [dict() for _ in range(vertex_count)]
    for u, v, capacity in edges:
        residual[u][v] = residual[u].get(v, 0) + capacity
        residual[v].setdefault(u, 0)

    total_flow = 0
    augmentations: list[tuple[list[int], int]] = []

    while True:
        parent = [-1] * vertex_count
        parent[source] = source
        queue = deque([source])

        while queue and parent[sink] == -1:
            u = queue.popleft()
            for v, remaining in residual[u].items():
                if remaining > 0 and parent[v] == -1:
                    parent[v] = u
                    queue.append(v)

        if parent[sink] == -1:
            return total_flow, augmentations

        # Reconstruct the BFS path and find its bottleneck.
        path = []
        bottleneck = float("inf")
        v = sink
        while v != source:
            u = parent[v]
            path.append(v)
            bottleneck = min(bottleneck, residual[u][v])
            v = u
        path.append(source)
        path.reverse()

        # Update both directions of every residual edge.
        v = sink
        while v != source:
            u = parent[v]
            residual[u][v] -= bottleneck
            residual[v][u] = residual[v].get(u, 0) + bottleneck
            v = u

        total_flow += bottleneck
        augmentations.append((path, bottleneck))


edges = [(0, 1, 4), (0, 2, 2), (1, 2, 1), (1, 3, 2), (2, 3, 3)]
value, paths = edmonds_karp(4, edges, 0, 3)
assert value == 5
assert sum(amount for _, amount in paths) == value
print(value, paths)
~~~
</details>

Each BFS costs $O(E)$. There are at most $O(VE)$ augmentations because BFS levels only increase in the critical-edge argument, giving total time $O(VE^2)$ and residual storage $O(V+E)$. This bound is polynomial and independent of the numeric maximum-flow value, though faster algorithms are preferable on large graphs.

**Practice:** [LeetCode 1349 - Maximum Students Taking Exam](https://leetcode.com/problems/maximum-students-taking-exam/) can be reduced to a bipartite conflict graph and then solved through matching or maximum flow; Edmonds-Karp is a clear baseline for testing that model.


### **Bipartite Matching** {#bipartite-matching}

A graph is **bipartite** when its vertices can be divided into disjoint sets $L$ and $R$ and every edge joins one side to the other. A **matching** is a set of edges in which no vertex appears more than once. The goal of maximum-cardinality matching is to pair as many vertices as possible.

Examples include assigning applicants to jobs, students to projects, or requests to compatible resources. A locally greedy rule can fail: matching one flexible applicant first may consume the only option available to another applicant. An **augmenting path** repairs this by alternating between unmatched and matched edges. Flipping membership along such a path increases the matching size by exactly one.

The flow reduction makes this structure explicit:

1. Add a source $s$ with capacity-$1$ edges to every $u\in L$.
2. Direct each candidate edge from $L$ to $R$ with capacity $1$.
3. Connect every $v\in R$ to sink $t$ with capacity $1$.

Because all capacities are integers, a maximum flow has an integral optimum. One unit through $s\rightarrow u\rightarrow v\rightarrow t$ selects pair $(u,v)$, and the outer unit capacities prevent either endpoint from being reused.

~~~text
MAXIMUM-BIPARTITE-MATCHING(L, adjacency)
    partner[right_vertex] <- unmatched for every right vertex

    for each left vertex u
        clear the right-side visited set for this attempt
        if AUGMENT(u)
            matching_size <- matching_size + 1

AUGMENT(u)
    for each candidate v adjacent to u
        if v was already visited in this attempt: continue
        mark v visited
        if v is unmatched or AUGMENT(partner[v]) succeeds
            partner[v] <- u
            return true
    return false
~~~

![A bipartite matching transformed into a unit-capacity source-to-sink flow network.](assets/bipartite-matching-flow-reduction.svg){fig-align="center" width="100%"}

<details>
<summary>Python implementation: augmenting-path bipartite matching</summary>

~~~python
def maximum_bipartite_matching(
    left_vertices: list[str],
    adjacency: dict[str, list[str]],
) -> set[tuple[str, str]]:
    right_partner: dict[str, str] = {}

    def augment(left: str, visited_right: set[str]) -> bool:
        for right in adjacency.get(left, []):
            if right in visited_right:
                continue
            visited_right.add(right)

            # Either claim an unused right vertex or reroute its current partner.
            if right not in right_partner or augment(
                right_partner[right], visited_right
            ):
                right_partner[right] = left
                return True
        return False

    for left in left_vertices:
        augment(left, set())

    return {(left, right) for right, left in right_partner.items()}


adjacency = {
    "L1": ["R1", "R2"],
    "L2": ["R1", "R3"],
    "L3": ["R2", "R3"],
}
matching = maximum_bipartite_matching(["L1", "L2", "L3"], adjacency)
assert len(matching) == 3
assert len({left for left, _ in matching}) == 3
assert len({right for _, right in matching}) == 3
print(matching)
~~~
</details>

The DFS augmenting-path implementation takes $O(|L|E)$ time: it attempts one search for each left vertex and may inspect all candidate edges. It uses $O(V+E)$ representation space and $O(V)$ recursion/visited space. Hopcroft-Karp groups many shortest augmenting paths into phases and improves the bound to $O(E\sqrt{V})$.

Konig's theorem adds a useful certificate: in every bipartite graph, maximum matching size equals minimum vertex-cover size. This is a matching-specific counterpart to max-flow min-cut.

**Practice:** [LeetCode 1947 - Maximum Compatibility Score Sum](https://leetcode.com/problems/maximum-compatibility-score-sum/) is a weighted assignment problem. It helps distinguish ordinary maximum-cardinality matching from the weighted version, which needs DP, Hungarian matching, or minimum-cost flow.


### **Circulations with Demands and Lower Bounds** {#circulations-with-demands-and-lower-bounds}

A **circulation** has no distinguished source or sink. Every vertex obeys a prescribed balance, and flow may move around cycles. This is the natural model for recurring transfers, supply chains with required shipments, and feasibility questions in which the first task is not to maximize throughput but to satisfy all obligations.

Let each edge have a lower bound $\ell(u,v)$ and upper bound $u(u,v)$:

$$
\ell(u,v)\le f(u,v)\le u(u,v).
$$

The lower bound is flow that must be sent, not optional capacity. Let $b(v)$ denote required net inflow at vertex $v$:

$$
\sum_{(u,v)\in E} f(u,v)-\sum_{(v,w)\in E} f(v,w)=b(v).
$$

A positive $b(v)$ is demand, a negative value is supply, and feasibility requires $\sum_v b(v)=0$. Pure circulation uses $b(v)=0$ for every vertex.

Lower bounds are removed by preloading them. Set $f(u,v)=\ell(u,v)$ and replace the optional capacity by $u(u,v)-\ell(u,v)$. The preload may violate vertex balances. If $L(v)$ is the preload's net inflow, the residual flow must contribute

$$
need(v)=b(v)-L(v).
$$

If $need(v)<0$, the vertex must send out $-need(v)$ additional units, so add an edge from super source $SS$ to $v$. If $need(v)>0$, the vertex must receive $need(v)$ units, so add $v\rightarrow TT$ to a super sink. A feasible circulation exists exactly when maximum flow from $SS$ to $TT$ saturates every edge leaving $SS$.

~~~text
FEASIBLE-CIRCULATION(edges with [lower, upper], vertex_demands b)
    preload every edge with its lower bound
    replace each capacity by upper - lower
    compute lower-bound net inflow L(v) at every vertex
    need(v) <- b(v) - L(v)

    if need(v) < 0: add SS -> v with capacity -need(v)
    if need(v) > 0: add v -> TT with capacity need(v)

    run maximum flow from SS to TT
    if every SS edge is saturated
        add residual flow to the preloaded lower bounds and return it
    return infeasible
~~~

![Lower bounds are preloaded, residual capacities become upper minus lower, and super nodes route the imbalance.](assets/circulation-lower-bound-transformation.svg){fig-align="center" width="100%"}

<details>
<summary>Python implementation: feasible circulation with lower bounds and demands</summary>

~~~python
from collections import deque
from dataclasses import dataclass


@dataclass
class _ResidualEdge:
    to: int
    reverse_index: int
    capacity: int


def feasible_circulation(
    vertex_count: int,
    bounded_edges: list[tuple[int, int, int, int]],
    required_net_inflow: list[int] | None = None,
) -> list[int] | None:
    demand = required_net_inflow or [0] * vertex_count
    if len(demand) != vertex_count or sum(demand) != 0:
        return None

    super_source = vertex_count
    super_sink = vertex_count + 1
    graph: list[list[_ResidualEdge]] = [[] for _ in range(vertex_count + 2)]

    def add_edge(u: int, v: int, capacity: int) -> int:
        forward_index = len(graph[u])
        reverse_index = len(graph[v])
        graph[u].append(_ResidualEdge(v, reverse_index, capacity))
        graph[v].append(_ResidualEdge(u, forward_index, 0))
        return forward_index

    lower_balance = [0] * vertex_count  # preload inflow minus outflow
    references: list[tuple[int, int, int, int]] = []

    for u, v, lower, upper in bounded_edges:
        if not 0 <= lower <= upper:
            raise ValueError("every edge must satisfy 0 <= lower <= upper")
        optional_capacity = upper - lower
        edge_index = add_edge(u, v, optional_capacity)
        references.append((u, edge_index, lower, optional_capacity))
        lower_balance[u] -= lower
        lower_balance[v] += lower

    required_from_super_source = 0
    for vertex in range(vertex_count):
        need = demand[vertex] - lower_balance[vertex]
        if need < 0:
            add_edge(super_source, vertex, -need)
            required_from_super_source += -need
        elif need > 0:
            add_edge(vertex, super_sink, need)

    def max_flow(source: int, sink: int) -> int:
        total = 0
        while True:
            parent: list[tuple[int, int] | None] = [None] * len(graph)
            parent[source] = (source, -1)
            queue = deque([source])

            while queue and parent[sink] is None:
                u = queue.popleft()
                for edge_index, edge in enumerate(graph[u]):
                    if edge.capacity > 0 and parent[edge.to] is None:
                        parent[edge.to] = (u, edge_index)
                        queue.append(edge.to)

            if parent[sink] is None:
                return total

            bottleneck = float("inf")
            v = sink
            while v != source:
                u, edge_index = parent[v]
                bottleneck = min(bottleneck, graph[u][edge_index].capacity)
                v = u

            v = sink
            while v != source:
                u, edge_index = parent[v]
                edge = graph[u][edge_index]
                edge.capacity -= bottleneck
                graph[v][edge.reverse_index].capacity += bottleneck
                v = u
            total += bottleneck

    if max_flow(super_source, super_sink) != required_from_super_source:
        return None

    recovered_flow = []
    for u, edge_index, lower, optional_capacity in references:
        unused_optional = graph[u][edge_index].capacity
        recovered_flow.append(lower + optional_capacity - unused_optional)
    return recovered_flow


# The lower bounds alone are unbalanced, but optional capacity repairs the cycle.
edges = [(0, 1, 2, 4), (1, 2, 1, 3), (2, 0, 1, 5)]
flow = feasible_circulation(3, edges)
assert flow is not None

balance = [0, 0, 0]
for (u, v, lower, upper), amount in zip(edges, flow):
    assert lower <= amount <= upper
    balance[u] -= amount
    balance[v] += amount
assert balance == [0, 0, 0]

# Node 0 supplies three units and node 1 demands three units.
assert feasible_circulation(2, [(0, 1, 0, 5)], [-3, 3]) == [3]
print(flow)
~~~
</details>

The transformation adds two vertices and at most $V$ balancing edges, then performs one maximum-flow computation on $V+2$ vertices and $E+O(V)$ edges. Recovery scans the original edges once. The overall asymptotic cost is therefore the cost of the chosen max-flow algorithm on the transformed graph.

For an $s$-$t$ flow with lower bounds, add a sufficiently large edge $t\rightarrow s$, solve the resulting circulation feasibility problem, and then recover how much crosses that added edge.

**Practice:** [LeetCode 1601 - Maximum Number of Achievable Transfer Requests](https://leetcode.com/problems/maximum-number-of-achievable-transfer-requests/) directly exercises zero net balance at every vertex, the defining invariant of a pure circulation.


### **Reductions to Network Flow** {#reductions-to-network-flow}

A **reduction to network flow** translates another problem into vertices, directed edges, capacities, and sometimes costs or lower bounds. The goal is not to make a drawing that resembles the story. A correct reduction must preserve both feasibility and objective value:

1. every legal solution to the original problem must map to a feasible flow;
2. every relevant integral flow must map back to a legal original solution;
3. the flow value or cut capacity must equal the quantity being optimized.

This two-way mapping is the proof. Without it, a network may accidentally permit duplicated assignments, fractional choices, or routes forbidden by the original problem.

Common modeling gadgets include:

- **Edge-disjoint paths:** give every original edge capacity $1$; maximum integral flow counts simultaneously usable paths.
- **Vertex-disjoint paths:** split each vertex $v$ into $v_{in}\rightarrow v_{out}$ with capacity $1$, then redirect incoming edges to $v_{in}$ and outgoing edges from $v_{out}$.
- **Vertex capacities:** use the same split with capacity $b(v)$ on the internal edge.
- **Multiple sources or sinks:** connect a super source or super sink using capacities equal to each terminal's supply or demand.
- **Assignment:** use a unit-capacity bipartite network.
- **Closure and project selection:** place positive-profit and negative-cost choices on opposite sides of an $s$-$t$ cut, with effectively infinite edges enforcing dependencies.

~~~text
VERTEX-CAPACITY-REDUCTION(vertices, directed_edges, limit)
    for each vertex v
        create v_in and v_out
        add v_in -> v_out with capacity limit(v)

    for each original edge u -> v with capacity c
        add u_out -> v_in with capacity c

    return the transformed edge-capacity network
~~~

![Three reusable reduction gadgets: vertex splitting, super terminals, and unit-capacity assignment.](assets/flow-reduction-patterns.svg){fig-align="center" width="100%"}

<details>
<summary>Python implementation: build a vertex-capacitated flow network</summary>

~~~python
def reduce_vertex_capacities(
    vertices: list[str],
    directed_edges: list[tuple[str, str, int]],
    vertex_limit: dict[str, int],
) -> tuple[list[str], list[tuple[str, str, int]]]:
    """Split each vertex so an ordinary edge capacity enforces its limit."""
    transformed_vertices: list[str] = []
    transformed_edges: list[tuple[str, str, int]] = []

    for vertex in vertices:
        vertex_in = f"{vertex}_in"
        vertex_out = f"{vertex}_out"
        transformed_vertices.extend((vertex_in, vertex_out))
        transformed_edges.append(
            (vertex_in, vertex_out, vertex_limit[vertex])
        )

    for u, v, capacity in directed_edges:
        # Original edges leave u_out and enter v_in.
        transformed_edges.append((f"{u}_out", f"{v}_in", capacity))

    return transformed_vertices, transformed_edges


vertices = ["s", "a", "b", "t"]
edges = [("s", "a", 4), ("a", "b", 3), ("b", "t", 4)]
limits = {"s": 10, "a": 2, "b": 3, "t": 10}

split_vertices, split_edges = reduce_vertex_capacities(vertices, edges, limits)
assert ("a_in", "a_out", 2) in split_edges
assert ("a_out", "b_in", 3) in split_edges
assert len(split_vertices) == 2 * len(vertices)
assert len(split_edges) == len(vertices) + len(edges)
print(split_edges)
~~~
</details>

Vertex splitting doubles the number of vertices and adds one internal edge per original vertex, so the transformation takes $O(V+E)$ time and creates $2V$ vertices and $V+E$ edges. The max-flow complexity must be evaluated on these transformed sizes, not on the original graph alone.

**Practice:** [LeetCode 1568 - Minimum Number of Days to Disconnect Island](https://leetcode.com/problems/minimum-number-of-days-to-disconnect-island/) develops vertex-cut modeling intuition; node splitting is the standard way to express general vertex capacities as edge capacities.


### **Comparison and Selection** {#flow-comparison-and-selection}

Network flow is appropriate when many interchangeable units compete for shared capacities and may use multiple routes. It is not a replacement for every graph algorithm. If the task asks for one cheapest route, use a shortest-path algorithm. If each edge also has a cost and the objective is to route flow as cheaply as possible, use **minimum-cost flow**. If the graph is specifically bipartite and only cardinality matters, a matching-specific algorithm may be simpler and faster.

For large general networks, **Dinic's algorithm** is a common practical default. It uses BFS to build a **level graph** containing only residual edges from level $i$ to $i+1$, then DFS to send a **blocking flow** that saturates at least one edge on every remaining source-to-sink path in that level graph. One BFS phase therefore accomplishes more than Edmonds-Karp's single augmentation.

~~~text
DINIC(G, capacity, source, sink)
    initialize paired forward and reverse residual edges
    total <- 0

    while BFS can assign a level to sink
        next_edge[v] <- 0 for every vertex
        while DFS can send positive flow through level-increasing edges
            update paired residual capacities
            add the pushed amount to total

    return total
~~~

![A decision workflow separating path, flow, matching, circulation, and cost-sensitive models.](assets/flow-algorithm-selection.svg){fig-align="center" width="100%"}

<details>
<summary>Python implementation: Dinic maximum flow</summary>

~~~python
from collections import deque
from dataclasses import dataclass


@dataclass
class Edge:
    to: int
    reverse_index: int
    capacity: int


def dinic_max_flow(
    vertex_count: int,
    edges: list[tuple[int, int, int]],
    source: int,
    sink: int,
) -> int:
    graph: list[list[Edge]] = [[] for _ in range(vertex_count)]

    def add_edge(u: int, v: int, capacity: int) -> None:
        forward = Edge(v, len(graph[v]), capacity)
        reverse = Edge(u, len(graph[u]), 0)
        graph[u].append(forward)
        graph[v].append(reverse)

    for u, v, capacity in edges:
        add_edge(u, v, capacity)

    total_flow = 0
    while True:
        level = [-1] * vertex_count
        level[source] = 0
        queue = deque([source])

        while queue:
            u = queue.popleft()
            for edge in graph[u]:
                if edge.capacity > 0 and level[edge.to] == -1:
                    level[edge.to] = level[u] + 1
                    queue.append(edge.to)

        if level[sink] == -1:
            return total_flow

        next_edge = [0] * vertex_count

        def send(u: int, available: int) -> int:
            if u == sink:
                return available

            while next_edge[u] < len(graph[u]):
                edge = graph[u][next_edge[u]]
                if edge.capacity > 0 and level[edge.to] == level[u] + 1:
                    pushed = send(edge.to, min(available, edge.capacity))
                    if pushed > 0:
                        edge.capacity -= pushed
                        graph[edge.to][edge.reverse_index].capacity += pushed
                        return pushed
                next_edge[u] += 1
            return 0

        while True:
            pushed = send(source, float("inf"))
            if pushed == 0:
                break
            total_flow += pushed


edges = [(0, 1, 4), (0, 2, 2), (1, 2, 1), (1, 3, 2), (2, 3, 3)]
assert dinic_max_flow(4, edges, 0, 3) == 5
print(dinic_max_flow(4, edges, 0, 3))
~~~
</details>

| Method | Path rule / structure | Typical guarantee | Best use |
|---|---|---:|---|
| Ford-Fulkerson with DFS | any residual path | $O(EF)$ for integer capacities | small graphs with small flow value |
| Edmonds-Karp | shortest residual path by edges | $O(VE^2)$ | teaching, deterministic baseline, debugging |
| Dinic | level graph plus blocking flow | $O(V^2E)$ in general | practical general-purpose maximum flow |
| Hopcroft-Karp | batches shortest alternating paths | $O(E\sqrt{V})$ | unweighted bipartite matching |
| Lower-bound transformation | super source/sink plus one max flow | cost of transformed max flow | feasibility with demands or mandatory flow |

The $F$ in Ford-Fulkerson is the numeric maximum-flow value. Dinic's general bound is polynomial in graph size and is often much faster in practice; specialized unit-capacity bounds are better still. Complexity should always use the transformed graph size when a reduction adds split vertices or super nodes.

Before implementation, ask:

- Is the objective one route or total simultaneous throughput?
- Are capacities directed, integral, and non-negative?
- Do vertices also have capacities, demands, or lower bounds?
- Must the actual edge flows, matching pairs, or minimum cut be reconstructed?
- Are edge costs part of the objective?

**Practice:** [LeetCode 1514 - Path with Maximum Probability](https://leetcode.com/problems/path-with-maximum-probability/) is intentionally a neighboring non-flow problem. It tests whether you can recognize that the objective concerns one multiplicative path, so a Dijkstra-style method is appropriate instead of maximum flow.
